In [ ]:
# import libraries
try:
  # %tensorflow_version only exists in Colab.
  !pip install tf-nightly
except Exception:
  pass
import tensorflow as tf
import pandas as pd
from tensorflow import keras
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import pandas as pd

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

train_df = pd.read_csv(train_file_path, sep='\t', header=None, names=['label', 'message'])
test_df = pd.read_csv(test_file_path, sep='\t', header=None, names=['label', 'message'])

max_tokens = 1000
sequence_length = 50

vectorize_layer = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode='int',
    output_sequence_length=sequence_length
)

train_texts = train_df['message'].values
train_labels = [1.0 if x == 'spam' else 0.0 for x in train_df['label'].values]

test_texts = test_df['message'].values
test_labels = [1.0 if x == 'spam' else 0.0 for x in test_df['label'].values]

vectorize_layer.adapt(train_texts)

train_dataset = tf.data.Dataset.from_tensor_slices((train_texts, train_labels)).shuffle(len(train_texts)).batch(64)
test_dataset = tf.data.Dataset.from_tensor_slices((test_texts, test_labels)).batch(64)

# Покращена модель з використанням Bidirectional LSTM для підвищення точності
model = tf.keras.Sequential([
    vectorize_layer,
    layers.Embedding(max_tokens, 64, mask_zero=True),
    layers.Bidirectional(layers.LSTM(32)),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_dataset,
    epochs=5,
    validation_data=test_dataset,
    verbose=1
)

def predict_message(pred_text):
    prediction = model.predict(tf.constant([pred_text]), verbose=0)[0][0]
    label = "spam" if prediction >= 0.5 else "ham"
    return [float(prediction), label]

print("Все успішно натреновано! Перевірка:", predict_message("lottery cash prize"))

In [ ]:
# Переконайтеся, що ви спочатку запустили попередній осередок (g_h508FEClxO), який визначає train_dataset та test_dataset!
import tensorflow as tf
from tensorflow.keras import layers

model = tf.keras.Sequential([
    vectorize_layer,
    layers.Embedding(max_tokens, 64, mask_zero=False),
    layers.GlobalAveragePooling1D(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_dataset,
    epochs=10,
    validation_data=test_dataset
)

In [ ]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):
    # Wrap input in a TensorFlow constant tensor to make it fully compatible with Keras 3
    prediction = model.predict(tf.constant([pred_text]), verbose=0)[0][0]

    label = "spam" if prediction >= 0.5 else "ham"

    return [float(prediction), label]

print(predict_message("how are you doing today?"))

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):

    try:
      prediction = predict_message(msg)
    except Exception:

      pred_val = model.predict(tf.constant([msg]), verbose=0)[0][0]
      label = "spam" if pred_val >= 0.5 else "ham"
      prediction = [float(pred_val), label]

    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()